In [1]:
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.datasets import make_friedman1

# 1. Setup Data & Parameters
X, y = make_friedman1(n_samples=1500, n_features=5, random_state=42)
dtrain = xgb.DMatrix(X, label=y)

params = {
    "objective": "reg:quantileerror",
    "quantile_alpha": [0.1, 0.5, 0.9],
    "tree_method": "hist",
    "max_depth": 5,
    "learning_rate": 0.05
}

# 2. Run CV to find the sweet spot
cv_results = xgb.cv(
    params=params,
    dtrain=dtrain,
    num_boost_round=1000,
    nfold=5,
    early_stopping_rounds=30,
    metrics="quantile",
    as_pandas=True,
    seed=42
)

# 3. Pull the optimal number of trees found during cross-validation
best_iteration = len(cv_results)

# 4. NOW WE TRAIN THE ACTUAL MODEL using that ideal tree count
model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=best_iteration
)

# 5. Predict on new data
# (For this example, we'll just predict on the training data)
preds = model.predict(dtrain)

# 6. Sort and map to columns
preds_sorted = np.sort(preds, axis=1)

df_intervals = pd.DataFrame(
    preds_sorted, 
    columns=['Lower_10', 'Median_50', 'Upper_90']
)

print(df_intervals.head())

    Lower_10  Median_50   Upper_90
0  15.716344  16.846754  16.865465
1  12.462784  12.518503  12.521604
2   5.863720   5.881758   8.484411
3   7.348210   7.587189   9.307315
4   8.718021   9.416162  10.502348
